# Дашборд конверсий

Ноутбук выполняет полный расчет проекта: загружает визиты и регистрации из API, рассчитывает конверсию, объединяет данные с `ads.csv` и сохраняет итоговые JSON и PNG-графики.


In [ ]:
import os

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

API_URL = os.getenv("API_URL")
DATE_BEGIN = os.getenv("DATE_BEGIN")
DATE_END = os.getenv("DATE_END")

if not all([API_URL, DATE_BEGIN, DATE_END]):
    raise RuntimeError("Не заданы API_URL, DATE_BEGIN или DATE_END")

os.makedirs("./charts", exist_ok=True)

print(API_URL, DATE_BEGIN, DATE_END)


## Загрузка данных из API

In [ ]:
params = {"begin": DATE_BEGIN, "end": DATE_END}

visits_response = requests.get(
    f"{API_URL}/visits",
    params=params,
    timeout=120,
)
visits_response.raise_for_status()

registrations_response = requests.get(
    f"{API_URL}/registrations",
    params=params,
    timeout=120,
)
registrations_response.raise_for_status()

visits = pd.DataFrame(visits_response.json())
registrations = pd.DataFrame(registrations_response.json())

print("visits:", visits.shape)
print("registrations:", registrations.shape)


## Расчет конверсии

In [ ]:
visits["datetime"] = pd.to_datetime(visits["datetime"], errors="raise")
registrations["datetime"] = pd.to_datetime(registrations["datetime"], errors="raise")

# Боты не должны влиять на конверсию.
visits = visits.loc[
    ~visits["user_agent"].str.contains("bot", case=False, na=False)
].copy()

# Для каждого visit_id учитываем только последний визит.
visits = (
    visits
    .sort_values("datetime")
    .drop_duplicates(subset="visit_id", keep="last")
)

visits["date_group"] = visits["datetime"].dt.normalize()
registrations["date_group"] = registrations["datetime"].dt.normalize()

visits_grouped = (
    visits
    .groupby(["date_group", "platform"], as_index=False)
    .agg(visits=("visit_id", "size"))
)

registrations_grouped = (
    registrations
    .groupby(["date_group", "platform"], as_index=False)
    .agg(registrations=("user_id", "size"))
)

conversion = visits_grouped.merge(
    registrations_grouped,
    on=["date_group", "platform"],
    how="outer",
)

conversion[["visits", "registrations"]] = (
    conversion[["visits", "registrations"]]
    .fillna(0)
    .astype(int)
)

conversion["conversion"] = (
    conversion["registrations"]
    .div(conversion["visits"])
    .mul(100)
    .where(conversion["visits"].ne(0), 0)
)

conversion = (
    conversion[
        ["date_group", "platform", "visits", "registrations", "conversion"]
    ]
    .sort_values(["date_group", "platform"])
    .reset_index(drop=True)
)

conversion.to_json("./conversion.json")
conversion.head()


## Рекламные кампании

In [ ]:
ads = pd.read_csv("./ads.csv")

ads["date"] = pd.to_datetime(ads["date"], errors="raise")
ads["date_group"] = ads["date"].dt.normalize()
ads["cost"] = pd.to_numeric(ads["cost"], errors="raise")

period_begin = pd.Timestamp(DATE_BEGIN)
period_end = pd.Timestamp(DATE_END)

# API использует правую границу периода как исключающую.
ads = ads.loc[
    (ads["date_group"] >= period_begin)
    & (ads["date_group"] < period_end)
].copy()

daily_conversion = (
    conversion
    .groupby("date_group", as_index=False)
    .agg(
        visits=("visits", "sum"),
        registrations=("registrations", "sum"),
    )
)

ads_grouped = (
    ads
    .groupby(["date_group", "utm_campaign"], as_index=False)
    .agg(cost=("cost", "sum"))
)

ads_result = daily_conversion.merge(
    ads_grouped,
    on="date_group",
    how="left",
)

ads_result["cost"] = ads_result["cost"].fillna(0)
ads_result["utm_campaign"] = ads_result["utm_campaign"].fillna("none")

ads_result = (
    ads_result[
        ["date_group", "visits", "registrations", "cost", "utm_campaign"]
    ]
    .sort_values("date_group")
    .reset_index(drop=True)
)

ads_result.to_json("./ads.json")
ads_result.head()


## Подготовка данных для графиков

In [ ]:
daily_metrics = (
    conversion
    .groupby("date_group", as_index=False)
    .agg(
        visits=("visits", "sum"),
        registrations=("registrations", "sum"),
    )
)

daily_metrics["conversion"] = (
    daily_metrics["registrations"]
    .div(daily_metrics["visits"])
    .mul(100)
    .where(daily_metrics["visits"].ne(0), 0)
)

visits_by_platform = (
    conversion
    .pivot(index="date_group", columns="platform", values="visits")
    .fillna(0)
)

registrations_by_platform = (
    conversion
    .pivot(index="date_group", columns="platform", values="registrations")
    .fillna(0)
)

ads_daily = (
    ads_result
    .groupby("date_group", as_index=False)
    .agg(cost=("cost", "sum"))
)

campaign_ranges = (
    ads.loc[:, ["date_group", "utm_campaign"]]
    .drop_duplicates()
    .groupby("utm_campaign", as_index=False)
    .agg(
        start=("date_group", "min"),
        end=("date_group", "max"),
    )
)


## Итоговые визиты

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily_metrics["date_group"], daily_metrics["visits"])
ax.axhline(daily_metrics["visits"].mean(), linestyle="--", label="mean")
ax.set_title("Total visits")
ax.set_xlabel("Date")
ax.set_ylabel("Visits")
ax.grid(axis="y")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig("./charts/total_visits.png")
plt.close(fig)


## Визиты по платформам

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
visits_by_platform.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("Visits by platform")
ax.set_xlabel("Date")
ax.set_ylabel("Visits")
ax.grid(axis="y")
ax.tick_params(axis="x", labelrotation=90)
fig.tight_layout()
fig.savefig("./charts/visits_by_platform.png")
plt.close(fig)


## Итоговые регистрации

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily_metrics["date_group"], daily_metrics["registrations"])
ax.axhline(
    daily_metrics["registrations"].mean(),
    linestyle="--",
    label="mean",
)
ax.set_title("Total registrations")
ax.set_xlabel("Date")
ax.set_ylabel("Registrations")
ax.grid(axis="y")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig("./charts/total_registrations.png")
plt.close(fig)


## Регистрации по платформам

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
registrations_by_platform.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("Registrations by platform")
ax.set_xlabel("Date")
ax.set_ylabel("Registrations")
ax.grid(axis="y")
ax.tick_params(axis="x", labelrotation=90)
fig.tight_layout()
fig.savefig("./charts/registrations_by_platform.png")
plt.close(fig)


## Конверсия по платформам

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for platform, data in conversion.groupby("platform"):
    ax.plot(
        data["date_group"],
        data["conversion"],
        label=platform,
    )

ax.set_title("Conversion by platform")
ax.set_xlabel("Date")
ax.set_ylabel("Conversion, %")
ax.grid(axis="y")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig("./charts/conversion_by_platform.png")
plt.close(fig)


## Средняя конверсия

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily_metrics["date_group"], daily_metrics["conversion"])
ax.axhline(
    daily_metrics["conversion"].mean(),
    linestyle="--",
    label="mean",
)
ax.set_title("Average conversion")
ax.set_xlabel("Date")
ax.set_ylabel("Conversion, %")
ax.grid(axis="y")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig("./charts/average_conversion.png")
plt.close(fig)


## Стоимость рекламы

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(ads_daily["date_group"], ads_daily["cost"])
ax.set_title("Advertising cost")
ax.set_xlabel("Date")
ax.set_ylabel("Cost")
ax.grid(axis="y")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig("./charts/ads_cost.png")
plt.close(fig)


## Визиты и рекламные кампании

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily_metrics["date_group"], daily_metrics["visits"], label="visits")

cmap = plt.get_cmap("tab10")
for index, row in campaign_ranges.iterrows():
    ax.axvspan(
        row["start"],
        row["end"] + pd.Timedelta(days=1),
        alpha=0.15,
        color=cmap(index % 10),
        label=row["utm_campaign"],
    )

ax.set_title("Visits during advertising campaigns")
ax.set_xlabel("Date")
ax.set_ylabel("Visits")
ax.grid(axis="y")
ax.legend(fontsize="small")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig("./charts/visits_with_ads.png")
plt.close(fig)


## Регистрации и рекламные кампании

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    daily_metrics["date_group"],
    daily_metrics["registrations"],
    label="registrations",
)

cmap = plt.get_cmap("tab10")
for index, row in campaign_ranges.iterrows():
    ax.axvspan(
        row["start"],
        row["end"] + pd.Timedelta(days=1),
        alpha=0.15,
        color=cmap(index % 10),
        label=row["utm_campaign"],
    )

ax.set_title("Registrations during advertising campaigns")
ax.set_xlabel("Date")
ax.set_ylabel("Registrations")
ax.grid(axis="y")
ax.legend(fontsize="small")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig("./charts/registrations_with_ads.png")
plt.close(fig)


Готово: `conversion.json`, `ads.json` и все PNG-графики сохраняются относительно директории запуска ноутбука.
